<a href="https://colab.research.google.com/github/WhatRaSudeep/SAiDL-Spring-Assignment-2024/blob/main/Graph%20Neural%20Networks/GAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import torch
os.environ['TORCH'] = torch.__version__
print(torch.__version__)

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git
import torch
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree, to_dense_adj, softmax
import torch_scatter



2.5.1+cu121
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='/tmp/Cora', name='Cora')
print(len(dataset))
print(dataset[0].num_nodes)
print(dataset[0].num_features)
print(dataset.num_classes)
print(dataset.num_node_features)

Processing...


1
2708
1433
7
1433


Done!


In [4]:
#GCN
import torch
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

class GCNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # "Add" aggregation (Step 5).
        self.lin = Linear(in_channels, out_channels, bias=False)
        self.bias = Parameter(torch.empty(out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()
        self.bias.data.zero_()

    def forward(self, x, edge_index):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype) # returns the degree of all nodes
        deg_inv_sqrt = deg.pow(-0.5)# inverse square root of the degree of all nodes
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0 #dealing with inf cases
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]# norm will be the weight matrix of len= num(edges)

        #So in essense you took the edge index, took individual edges from it, calculated the degrees of both the ends of the edge and you multiplied
        #inverse square roots together to get a vector of shape [E]

        # Step 4-5: Start propagating messages.
        out = self.propagate(edge_index, x=x, norm=norm)

        # Step 6: Apply a final bias vector.
        out = out + self.bias

        return out

    def message(self, x_j, norm):
        # x_j has shape [E, out_channels]

        # Step 4: Normalize node features.
        return norm.view(-1, 1) * x_j #This is your aggregated message that you are going to pass on.

In [12]:
class GAT(MessagePassing):

    def __init__(self, in_channels, out_channels, heads = 1,
                 negative_slope = 0.2, dropout = 0., **kwargs):
        super(GAT, self).__init__(node_dim=0, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.negative_slope = negative_slope
        self.dropout = dropout

        self.lin_l = None
        self.lin_r = None
        self.att_l = None
        self.att_r = None

        self.lin_l = Linear(in_channels, out_channels*self.heads)
        self.lin_r = self.lin_l
        self.att_l = Parameter(torch.Tensor(1, self.heads, out_channels))
        self.att_r = Parameter(torch.Tensor(1, self.heads, out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin_l.weight)
        nn.init.xavier_uniform_(self.lin_r.weight)
        nn.init.xavier_uniform_(self.att_l)
        nn.init.xavier_uniform_(self.att_r)

    def forward(self, x, edge_index, size = None):

        H, C = self.heads, self.out_channels

        wh_l = self.lin_l(x).view(-1, H, C)
        wh_r = self.lin_r(x).view(-1, H, C)
        alpha_l = torch.mul(self.att_l, wh_l)
        alpha_r = torch.mul(self.att_r, wh_r)
        out = self.propagate(edge_index,x=(wh_l, wh_r), size=size, alpha=(alpha_l, alpha_r))
        out = out.view(-1, H*C)
        return out


    def message(self, x_j, alpha_j, alpha_i, index, ptr, size_i):

        att = alpha_i + alpha_j

        att = F.leaky_relu(att, negative_slope=self.negative_slope)
        att = softmax(att, ptr if ptr else index)
        att = F.dropout(att, self.dropout)
        out = torch.mul(x_j, att)
        return out

    def aggregate(self, inputs, index, dim_size = None):
        out = torch_scatter.scatter(inputs, index = index, dim = self.node_dim, dim_size = dim_size, reduce = "sum")

        return out



In [6]:
import torch.nn as nn
class GAT_MLP_Layer(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=1, negative_slope=0.2, dropout=0.0):
        super().__init__(aggr="add")  # Aggregation method (e.g., sum, mean)

        self.heads = heads
        self.out_channels = out_channels
        self.dropout = dropout
        self.negative_slope = negative_slope

        # Linear transformation for input features
        self.lin = Linear(in_channels, heads * out_channels, bias=False)

        # Attention MLP
        self.att_mlp = nn.Sequential(
            Linear(2 *heads*out_channels, 512, bias=False),
            nn.ReLU(),
            Linear(512, 1, bias=False)
        )


        # Final projection (if needed)
        self.out_proj = Linear(heads * out_channels, out_channels, bias=False)

    def forward(self, x, edge_index):
        # Apply linear transformation and reshape for multi-head attention
        x = self.lin(x).view(-1, self.heads, self.out_channels)

        # Perform message passing
        out = self.propagate(edge_index, x=x)

        # Concatenate heads and apply output projection
        out = out.view(-1, self.heads * self.out_channels)
        out = self.out_proj(out)
        return out

    def message(self, x_j, x_i):
        # Concatenate features from source (x_j) and target (x_i)
        x_ij = torch.cat([x_i, x_j], dim=-1)  # Shape: [num_edges, 2 * head* out_channels]

        # Compute unnormalized attention scores
        alpha = self.att_mlp(x_ij).squeeze(-1)  # Shape: [num_edges]

        # Apply LeakyReLU and normalize
        alpha = F.leaky_relu(alpha, self.negative_slope)
        alpha = F.softmax(alpha, dim=0)

        # Apply dropout to attention coefficients
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)
        return x_j * alpha.unsqueeze(-1)  # Shape: [num_edges, out_channels]

    def update(self, aggr_out):
        # Final aggregation output per node
        return aggr_out


In [17]:
from torch import nn
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='/tmp/Cora', name='Cora')
import torch.nn.functional as F
class GNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GAT(dataset.num_node_features, 16)
        self.conv2 = GAT(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.log_softmax(x, dim=1)
        return x

model = GNN()
print(model)

GNN(
  (conv1): GAT(1433, 16)
  (conv2): GAT(16, 7)
)


In [18]:
data = dataset[0]
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay= 5e-4)
model.train()
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

Epoch: 000, Loss: 1.9555
Epoch: 001, Loss: 1.8663
Epoch: 002, Loss: 1.7441
Epoch: 003, Loss: 1.6128
Epoch: 004, Loss: 1.4357
Epoch: 005, Loss: 1.3255
Epoch: 006, Loss: 1.1919
Epoch: 007, Loss: 1.0413
Epoch: 008, Loss: 0.9374
Epoch: 009, Loss: 0.8484
Epoch: 010, Loss: 0.8191
Epoch: 011, Loss: 0.6970
Epoch: 012, Loss: 0.6275
Epoch: 013, Loss: 0.5262
Epoch: 014, Loss: 0.4795
Epoch: 015, Loss: 0.3755
Epoch: 016, Loss: 0.4168
Epoch: 017, Loss: 0.4268
Epoch: 018, Loss: 0.3347
Epoch: 019, Loss: 0.2896
Epoch: 020, Loss: 0.2623
Epoch: 021, Loss: 0.2873
Epoch: 022, Loss: 0.2759
Epoch: 023, Loss: 0.2954
Epoch: 024, Loss: 0.2387
Epoch: 025, Loss: 0.2347
Epoch: 026, Loss: 0.1793
Epoch: 027, Loss: 0.1988
Epoch: 028, Loss: 0.1741
Epoch: 029, Loss: 0.1730
Epoch: 030, Loss: 0.1208
Epoch: 031, Loss: 0.1435
Epoch: 032, Loss: 0.1378
Epoch: 033, Loss: 0.2007
Epoch: 034, Loss: 0.1135
Epoch: 035, Loss: 0.1487
Epoch: 036, Loss: 0.1039
Epoch: 037, Loss: 0.1114
Epoch: 038, Loss: 0.0750
Epoch: 039, Loss: 0.1062


In [19]:
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')


Accuracy: 0.7770


In [20]:
class GNN_gcnn(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNLayer(dataset.num_node_features, 16)
        self.conv2 = GCNLayer(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.log_softmax(x, dim=1)
        return x

model = GNN_gcnn()
data = dataset[0]
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay= 5e-4)

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')



Epoch: 000, Loss: 1.9457
Epoch: 001, Loss: 1.8959
Epoch: 002, Loss: 1.8213
Epoch: 003, Loss: 1.7366
Epoch: 004, Loss: 1.6464
Epoch: 005, Loss: 1.5396
Epoch: 006, Loss: 1.4183
Epoch: 007, Loss: 1.3383
Epoch: 008, Loss: 1.2173
Epoch: 009, Loss: 1.1259
Epoch: 010, Loss: 1.0105
Epoch: 011, Loss: 0.8965
Epoch: 012, Loss: 0.8224
Epoch: 013, Loss: 0.7455
Epoch: 014, Loss: 0.7076
Epoch: 015, Loss: 0.5663
Epoch: 016, Loss: 0.5291
Epoch: 017, Loss: 0.4485
Epoch: 018, Loss: 0.4185
Epoch: 019, Loss: 0.3876
Epoch: 020, Loss: 0.2994
Epoch: 021, Loss: 0.2841
Epoch: 022, Loss: 0.2594
Epoch: 023, Loss: 0.2732
Epoch: 024, Loss: 0.2537
Epoch: 025, Loss: 0.1668
Epoch: 026, Loss: 0.1627
Epoch: 027, Loss: 0.1417
Epoch: 028, Loss: 0.1468
Epoch: 029, Loss: 0.1615
Epoch: 030, Loss: 0.1406
Epoch: 031, Loss: 0.1269
Epoch: 032, Loss: 0.0951
Epoch: 033, Loss: 0.0965
Epoch: 034, Loss: 0.0902
Epoch: 035, Loss: 0.0890
Epoch: 036, Loss: 0.0716
Epoch: 037, Loss: 0.0927
Epoch: 038, Loss: 0.0653
Epoch: 039, Loss: 0.0646


In [21]:
class GNN_gat_mlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GAT_MLP_Layer(dataset.num_node_features, 16)
        self.conv2 = GAT_MLP_Layer(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.log_softmax(x, dim=1)
        return x

model = GNN_gcnn()
data = dataset[0]
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay= 5e-4)

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.9469
Epoch: 001, Loss: 1.8978
Epoch: 002, Loss: 1.8187
Epoch: 003, Loss: 1.7307
Epoch: 004, Loss: 1.6276
Epoch: 005, Loss: 1.5258
Epoch: 006, Loss: 1.4243
Epoch: 007, Loss: 1.3103
Epoch: 008, Loss: 1.2390
Epoch: 009, Loss: 1.1174
Epoch: 010, Loss: 1.0412
Epoch: 011, Loss: 0.9519
Epoch: 012, Loss: 0.8608
Epoch: 013, Loss: 0.7874
Epoch: 014, Loss: 0.6986
Epoch: 015, Loss: 0.6416
Epoch: 016, Loss: 0.5833
Epoch: 017, Loss: 0.5294
Epoch: 018, Loss: 0.4553
Epoch: 019, Loss: 0.4276
Epoch: 020, Loss: 0.3795
Epoch: 021, Loss: 0.3351
Epoch: 022, Loss: 0.3055
Epoch: 023, Loss: 0.2613
Epoch: 024, Loss: 0.2760
Epoch: 025, Loss: 0.2702
Epoch: 026, Loss: 0.2320
Epoch: 027, Loss: 0.1754
Epoch: 028, Loss: 0.1613
Epoch: 029, Loss: 0.1343
Epoch: 030, Loss: 0.1419
Epoch: 031, Loss: 0.1583
Epoch: 032, Loss: 0.1147
Epoch: 033, Loss: 0.1305
Epoch: 034, Loss: 0.1021
Epoch: 035, Loss: 0.1005
Epoch: 036, Loss: 0.1425
Epoch: 037, Loss: 0.0824
Epoch: 038, Loss: 0.0951
Epoch: 039, Loss: 0.0919
